#### Test#5: Binary Matrix Rank Test

In [1]:
import galois
import random
import numpy as np
from math import exp

In [2]:
def calculate_matrix_rank(matrix):
    GF2 = galois.GF(2)
    A = GF2(matrix) 
    return np.linalg.matrix_rank(A) 

def calculate_N_blocks(n, M, Q):
    N = n // (M * Q)
    return N

def create_blocks(string_bits, M, Q):
    N = calculate_N_blocks(len(string_bits), M, Q)
    blocks = []
    for i in range(N):
        block_bits = bits[i * M * Q:(i + 1) * M * Q]
        matrix = [block_bits[j*Q:(j+1)*Q] for j in range(M)]
        blocks.append(matrix)
    return blocks
    

def binary_matrix_rank_test(string_bits, M, Q):
    N = calculate_N_blocks(len(string_bits), M, Q)
    if N < 1:
        return "La secuencia es muy pequenna"
    
    blocks = create_blocks(string_bits, M, Q)
    full_rank = min(M, Q)
    rank_full = 0
    rank_full_minus1 = 0
    others = 0
    
    for block in blocks:
        rank = calculate_matrix_rank(block)
        if rank == full_rank:
            rank_full += 1
        elif rank == full_rank - 1:
            rank_full_minus1 += 1
        else:
            others += 1

    if M == 32 and Q == 32:
        p_full = 0.2888
        p_full_minus1 = 0.5776
        p_other = 0.1336
    else:
        raise NotImplementedError("Distribución teórica solo implementada para 32x32.")
    
    chi_sq = ((rank_full - p_full * N) ** 2) / (p_full * N) + \
             ((rank_full_minus1 - p_full_minus1 * N) ** 2) / (p_full_minus1 * N) + \
             ((others - p_other * N) ** 2) / (p_other * N)

    p_value = exp(-chi_sq / 2)

    return {
        "N": N,
        "rank_full": rank_full,
        "rank_full_minus1": rank_full_minus1,
        "others": others,
        "chi_sq": chi_sq,
        "p_value": p_value
    }

    

In [3]:
bits = [random.randint(0, 1) for _ in range(10000)]
result = binary_matrix_rank_test(bits, M=32, Q=32)
print(result)

{'N': 9, 'rank_full': 4, 'rank_full_minus1': 4, 'others': 1, 'chi_sq': 1.0652803349902225, 'p_value': 0.5870530035966469}


#### Test#8: Overlapping Template Matching Test

In [14]:
import math
from math import exp
from scipy.stats import chi2

In [16]:
def create_N_blocks(bits, M):
    N = len(bits) // M
    blocks = [bits[i*M:(i+1)*M] for i in range(N)]
    return blocks, N

def count_overlapping_matches(block, B):
    m = len(B)
    count = 0
    for i in range(len(block) - m + 1):
        if block[i:i+m] == B:
            count += 1
    return count

def compute_B_matches(blocks, B):
    return [count_overlapping_matches(block, B) for block in blocks]

def overlapping_template_matching_test(bits, B=None, M=1032, m=9):
    if B is None:
        B = [1]*m
    if len(B) != m:
        raise ValueError("La longitud del template B debe ser igual a m.")

    blocks, N = create_N_blocks(bits, M)
    if N == 0:
        raise ValueError("Secuencia demasiado corta para formar bloques.")

    counts = compute_B_matches(blocks, B)

    lam = (M - m + 1) / (2**m)   
    K = 5  
    
    pi = [math.exp(-lam) * lam**i / math.factorial(i) for i in range(K-1)]
    pi.append(1 - sum(pi))  

    nu = [0]*K
    for c in counts:
        if c <= K-2:
            nu[c] += 1
        else:
            nu[K-1] += 1

   
    for p in pi:
        if p == 0:
            raise ZeroDivisionError("Probabilidad teórica cero para una categoría; revise parámetros M y m.")

    chi_sq = sum((nu[i] - N*pi[i])**2 / (N*pi[i]) for i in range(K))
    p_value = 1 - chi2.cdf(chi_sq, K-1)

    return {
        "N": N,
        "lambda": lam,
        "counts": nu,
        "chi_sq": chi_sq,
        "p_value": p_value
    }

In [22]:
bits = [random.randint(0, 1) for _ in range(10000)]
print(bits)
resultado = overlapping_template_matching_test(bits)
print(resultado)


[1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 

In [23]:
bits = [1,1,1,1,0,1,1,1,0,0,1,1,1,1,1,0,0,1,1,1] 
res = overlapping_template_matching_test(bits, B=[1,0,1], M=20, m=3)
print(res)


{'N': 1, 'lambda': 2.25, 'counts': [0, 1, 0, 0, 0], 'chi_sq': 3.216771482826011, 'p_value': np.float64(0.5222263307266528)}


#### Test#12: Approximate Entropy Test: 

In [19]:
import math
from collections import Counter
from scipy.special import gammaincc 

In [7]:
def augment_bits_sequence(bits, m):
    return bits + bits[:m-1]

def get_frequency_counts(bits, m):
    n = len(bits) - (m - 1)
    blocks = [bits[i:i+m] for i in range(n)]
    counts = Counter(blocks)
    total = len(blocks)
    probabilities = {k: v / total for k, v in counts.items()}
    return probabilities

def compute_phi(probabilities, m):
    phi = 0.0
    for i in range(2**m):
        block = format(i, f'0{m}b')
        p = probabilities.get(block, 0)
        if p > 0:
            phi += p * math.log(p)
    return phi

def compute_apen_and_chi(bits, m):
    n = len(bits)
    bits_m = augment_bits_sequence(bits, m)
    bits_m1 = augment_bits_sequence(bits, m + 1)

    freq_m = get_frequency_counts(bits_m, m)
    freq_m1 = get_frequency_counts(bits_m1, m + 1)

    phi_m = compute_phi(freq_m, m)
    phi_m1 = compute_phi(freq_m1, m + 1)

    apen = phi_m - phi_m1
    chi_squared = 2 * n * (math.log(2) - apen)
    return apen, chi_squared

def compute_p_value(chi_squared, m):
    df = 2 ** (m - 1)
    p_value = gammaincc(df / 2, chi_squared / 2)
    return p_value

def approximate_entropy_test(bits, m):
    apen, chi_squared = compute_apen_and_chi(bits, m)
    p_value = compute_p_value(chi_squared, m)
    return {
        "ApEn": apen,
        "Chi-squared": chi_squared,
        "P-value": p_value
    }


In [58]:
#bits = ''.join(str(random.randint(0, 1)) for _ in range(10000))
bits = "1100100100001111110110101010001000100001011010001100001000110100110001001100011001100010100010111000"
result = approximate_entropy_test(bits, m=2)
print(result)

{'ApEn': 0.6653932193180652, 'Chi-squared': 5.550792248376024, 'P-value': np.float64(0.062324783444965406)}
